# CAP MVP — full downstream pipeline
Validate (ArcFace identity preservation) → Audit (DeepFace + InsightFace) → Analyze (H1/H2/H3 + flip rates) → Visualize
Run AFTER the MVP generation notebook (02) completes.

# 04 — Validate + Analyze (full stats) + Visualize

Closes the paper-quality gaps after `02_databricks_mvp_run.ipynb` (generate) and `03_databricks_audit_analyze_viz.ipynb` (audit):

1. **Validate** — ArcFace cosine similarity for each (seed, counterfactual) pair (NEW)
2. Skip audit (`predictions.parquet` already exists from run 03)
3. **Analyze** — full confirmatory tests: H1 ANOVA, H2 McNemar's, H3 ordinal logit, FDR-corrected, plus pairwise flip rate (NEW)
4. **Visualize** — intersectional heatmaps + interactive dashboard (now that intersectional CSVs land)

**Cluster:** same single-user cluster. Validate uses the driver L4 (~10 min for 600 images). Analyze + visualize are CPU-only (~2 min total).

## 1. Install cap

In [ ]:
%pip install ftfy regex torchsde tf-keras
%pip install /Workspace/Users/alenj00@safeway.com/counterfactual-audit-pipeline

In [ ]:
dbutils.library.restartPython()

## 2. Volume cache + HF token (validate uses InsightFace, needs cache)

In [ ]:
import os
from pathlib import Path

HF_CACHE = "/Volumes/ds_work/alenj00/cap_cache/hf_cache"
os.environ["HF_HOME"] = HF_CACHE
os.environ["HUGGINGFACE_HUB_CACHE"] = HF_CACHE
os.environ["TRANSFORMERS_CACHE"] = HF_CACHE
os.environ["HF_HUB_DISABLE_XET"] = "1"  # xet fails on GCS-FUSE Volumes

HF_TOKEN = dbutils.secrets.get(scope="cap-secrets", key="hf-token")
os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN

REPO = "/Workspace/Users/alenj00@safeway.com/counterfactual-audit-pipeline"
RUN_DIR = Path("/Volumes/ds_work/alenj00/cap_cache/runs/mvp")
GENERATED = RUN_DIR / "generated"
AUDIT = RUN_DIR / "audit"
VALIDATION = RUN_DIR / "validation"
ANALYSIS = RUN_DIR / "analysis"
VIZ = RUN_DIR / "viz"

assert (GENERATED / "manifest.parquet").exists(), "generation manifest missing — run 02 first"
# (audit predictions get written by the audit cell below, no preflight check)
print("prereqs OK; outputs will land in:", RUN_DIR)

In [ ]:
import pandas as pd

# Repair: the MVP run was generated before ray_runner.py flattened axis_values
# into per-axis columns, so manifest.parquet + predictions.parquet have an
# axis_values DICT column instead of axis_skin_tone / axis_gender. Expand it
# now so analyze.py can groupby on those keys. Idempotent — checks first.
def _ensure_axis_columns(parquet_path: Path) -> None:
    if not parquet_path.exists():
        print(f'(skip repair: {parquet_path} not yet written)')
        return
    df = pd.read_parquet(parquet_path)
    if "axis_values" not in df.columns:
        return
    if "axis_skin_tone" in df.columns and "axis_gender" in df.columns:
        return
    print(f"Repairing {parquet_path}: expanding axis_values dict...")
    expanded = pd.json_normalize(df["axis_values"]).add_prefix("axis_")
    df = pd.concat([df.drop(columns=["axis_values"]), expanded], axis=1)
    df.to_parquet(parquet_path)
    print(f"  → cols now: {[c for c in df.columns if c.startswith('axis_')]}")

_ensure_axis_columns(GENERATED / "manifest.parquet")
_ensure_axis_columns(AUDIT / "predictions.parquet")
print("Repair check done.")

## 3. Validate — ArcFace identity preservation

Computes cosine similarity between the seed FairFace image and each generated counterfactual using the antelopev2 ArcFace pack we already pre-staged on the Volume. Threshold from config (default 0.5). Output: `identity_scores.parquet` + `summary.json`.

In [ ]:
import time
from click.testing import CliRunner
from cap.cli.validate import main as validate_cli

runner = CliRunner()
t0 = time.time()
result = runner.invoke(
    validate_cli,
    [
        "--config", f"{REPO}/configs/mvp.yaml",
        "--cache-dir", HF_CACHE,  # antelopev2 ONNX files live here on the Volume
    ],
    catch_exceptions=False,
    standalone_mode=False,
)
wall = time.time() - t0
print(f"\nValidate wall: {wall:.0f}s ({wall/60:.1f} min)")
print("CLI output:")
print(result.output[:3000])

import json
summary_path = VALIDATION / "summary.json"
if summary_path.exists():
    print("\nIdentity validation summary:")
    print(json.dumps(json.loads(summary_path.read_text()), indent=2))

## Audit (DeepFace + InsightFace × all 600 images)
Two local auditors per Review 001 §4 follow-up. Cloud auditors plug in via configs/mvp.yaml when API credentials are available.

In [ ]:
import time
from click.testing import CliRunner
from cap.cli.audit import main as audit_cli

REPO = "/Workspace/Users/alenj00@safeway.com/counterfactual-audit-pipeline"

t0 = time.time()
runner = CliRunner()
result = runner.invoke(
    audit_cli,
    ["--config", f"{REPO}/configs/mvp.yaml"],
    catch_exceptions=False,
    standalone_mode=False,
)
wall = time.time() - t0
print(f"\nAudit wall: {wall:.0f}s ({wall/60:.1f} min)")
print("CLI output:")
print(result.output[:3000])

## 4. Analyze — full stats (H1 ANOVA, H2 McNemar's, H3 ordinal logit, FDR-corrected, plus pairwise flip rate)

In [ ]:
from cap.cli.analyze import main as analyze_cli

t0 = time.time()
result = runner.invoke(
    analyze_cli,
    ["--config", f"{REPO}/configs/mvp.yaml"],
    catch_exceptions=False,
    standalone_mode=False,
)
wall = time.time() - t0
print(f"Analyze wall: {wall:.0f}s")
print("CLI output:")
print(result.output[:5000])

print("\nAnalysis outputs:")
for p in sorted(ANALYSIS.glob("*")):
    print(f"  {p.name} ({p.stat().st_size} bytes)")

## 5. Stats results preview

In [ ]:
import pandas as pd

for csv in ["flip_rates.csv", "pairwise_flip_rates.csv", "h1_anova.csv", "h2_mcnemars.csv", "h3_ordinal_logit.csv"]:
    p = ANALYSIS / csv
    print(f"\n=== {csv} ===")
    if p.exists():
        try:
            print(pd.read_csv(p).to_string(index=False))
        except pd.errors.EmptyDataError:
            print("(empty)")
    else:
        print("(not produced)")

print("\n=== summary.json ===")
import json
summary = json.loads((ANALYSIS / "summary.json").read_text())
print(json.dumps(summary, indent=2)[:3000])

## 6. Visualize — figures + interactive dashboard

In [ ]:
from cap.cli.visualize import main as visualize_cli

t0 = time.time()
result = runner.invoke(
    visualize_cli,
    ["--config", f"{REPO}/configs/mvp.yaml"],
    catch_exceptions=False,
    standalone_mode=False,
)
wall = time.time() - t0
print(f"Visualize wall: {wall:.0f}s")
print("CLI output:")
print(result.output[:3000])

print("\nFigures:")
for p in sorted(VIZ.rglob("*")):
    if p.is_file():
        print(f"  {p.relative_to(VIZ)} ({p.stat().st_size} bytes)")

## 7. Visual preview

In [ ]:
from PIL import Image

# Sample one generated counterfactual
sample_png = sorted(GENERATED.glob("*.png"))[0]
print(f"Sample counterfactual: {sample_png.name}")
display(Image.open(sample_png))

# Sample any intersectional heatmap PNG
for hm in sorted((VIZ / "figures").glob("intersectional_*.png")):
    print(f"Heatmap: {hm.name}")
    display(Image.open(hm))
    break